# Capability & Registry Showcase

This notebook wires every core capability into a single agent:
- **Context** packs shared parameters and ad-hoc datasets.
- **Resources** provide knowledge bases the agent can cite.
- **Skill registry** injects callable skills for intra-agent logic.
- **Tools** (with permissions and budgets) demonstrate guarded execution.
- **Memory** captures episodic traces written during a run.
- **Message bus** exposes the cross-capability event stream.
- **Reinforcement learning** adjusts agent hyperparameters on the fly.

Each cell builds on the last so you can see the moving parts in action.

In [1]:
# Cell 1: imports and capability scaffolding

from typing import Any, Mapping

from agentic_codex import AgentBuilder, Context, FunctionAdapter, SkillRegistry
from agentic_codex.core.capabilities import ContextCapability, MessageBusCapability, ResourceCapability, ToolCapability
from agentic_codex.core.memory import EpisodicMemory
from agentic_codex.core.message_bus import MessageBus
from agentic_codex.core.schemas import AgentStep, Message
from agentic_codex.core.tools import MathToolAdapter, ToolPermissions, BudgetGuard

AGENT_NAME = "showcase"
BUS = MessageBus()
STUB_LLM = FunctionAdapter(lambda prompt: f"[stub llm] {prompt[-160:]}")

CONTEXT_CAPABILITY = ContextCapability(
    data={
        "parameters": {"temperature": 0.7, "tone": "friendly"},
        "dataset": {"columns": ["capability", "description"], "size": 6},
    }
)

KNOWLEDGE_CAPABILITY = ResourceCapability(
    name="knowledge.base",
    resource={
        "capabilities": [
            "Context packs share goals, data, and hyperparameters.",
            "Tools execute guarded operations like math or retrieval.",
            "Skill registries expose pure-python helpers to agents.",
            "Memories persist signals across iterations.",
        ],
        "tools": "Math tool is budget guarded; extend with HTTP/RAG as needed.",
    },
    target="components",
)

TOOLS = {"math": MathToolAdapter(name="math")}
TOOL_PERMISSIONS = ToolPermissions({AGENT_NAME: {"math"}})
TOOL_BUDGET = BudgetGuard(max_calls=5)
TOOL_CAPABILITY = ToolCapability(TOOLS, permissions=TOOL_PERMISSIONS, budget=TOOL_BUDGET)

RL_CONFIG = {"target_tool_result": 25.0, "step": 0.1}


In [3]:
# Cell 2: skills, reinforcement learning trainer, and agent step

skill_registry = SkillRegistry()

def greeting_skill(*, subject: str) -> str:
    return f"Greetings, {subject}!"

skill_registry.register("greeting", greeting_skill)

class ShowcaseTrainer:
    """Adjust the storytelling temperature based on tool feedback."""

    def setup(self, context: Context, *, environment: Mapping[str, Any] | None = None) -> None:
        context.scratch.setdefault("showcase_rl", [])

    def update(self, context: Context, step: AgentStep, *, environment: Mapping[str, Any] | None = None) -> None:
        env = environment or {}
        metrics = context.scratch.get("showcase_metrics", {})
        tool_result = float(metrics.get("tool_result", 0.0))
        target = float(env.get("target_tool_result", tool_result))
        step_size = float(env.get("step", 0.1))
        parameters = context.scratch["context"]["parameters"]
        temperature = float(parameters.get("temperature", 0.7))
        if tool_result < target:
            action = "increase"
            temperature = min(1.5, temperature + step_size)
        else:
            action = "decrease"
            temperature = max(0.1, temperature - step_size)
        parameters["temperature"] = temperature
        entry = {
            "iteration": context.scratch.get("iteration", 0),
            "tool_result": tool_result,
            "target": target,
            "action": action,
            "temperature": round(temperature, 3),
        }
        context.scratch.setdefault("showcase_rl", []).append(entry)
        try:
            bus = context.get_message_bus()
            bus.publish(
                agent="rl_showcase",
                content=f"[iter {entry['iteration']}] {action} temperature -> {temperature:.2f}",
                iteration=context.scratch.get("iteration", 0),
                channel="rl",
            )
        except KeyError:
            pass

trainer = ShowcaseTrainer()


def showcase_step(ctx: Context) -> AgentStep:
    ctx.scratch.setdefault("iteration", 0)
    bus = ctx.get_message_bus()
    parameters = ctx.scratch["context"]["parameters"]
    math_tool = ctx.get_tool("math")
    math_result = math_tool.invoke(expression="(2+3)*4")
    greeting = ctx.registry.get("greeting")(subject=ctx.goal)
    ctx.memory["default"].put("last_greeting", greeting)
    knowledge = ctx.get_component("knowledge.base")
    llm_text = ctx.llm.generate(
        "".join(
            [
                "You are an explainer agent summarising how capabilities combine.",
                f"Knowledge snippets: {knowledge['capabilities']}",
                f"Current parameters: {parameters}",
                f"Latest tool output: {math_result['result']}",
                f"Skill response: {greeting}",
                f"Goal: {ctx.goal}",
            ]
        )
    )
    ctx.scratch["showcase_metrics"] = {"tool_result": float(math_result["result"])}
    summary = {
        "greeting": greeting,
        "math_result": math_result["result"],
        "parameters": dict(parameters),
        "llm_excerpt": llm_text[:160],
    }
    bus.publish(
        agent="showcase_agent",
        content=llm_text,
        iteration=ctx.scratch.get("iteration", 0),
        channel="summary",
    )
    return AgentStep(
        out_messages=[Message(role="showcase_agent", content=llm_text)],
        state_updates={"summary": summary},
    )


In [4]:
# Cell 3: assemble the agent and execute multiple passes

showcase_agent = (
    AgentBuilder(name=AGENT_NAME, role="demo")
    .with_capabilities([MessageBusCapability(bus=BUS), CONTEXT_CAPABILITY, KNOWLEDGE_CAPABILITY])
    .with_capability(TOOL_CAPABILITY)
    .with_llm(STUB_LLM)
    .with_memory(EpisodicMemory())
    .with_skill_registry(skill_registry)
    .with_reinforcement_learning(trainer, environment=RL_CONFIG)
    .with_step(showcase_step)
    .build()
)

context = Context(goal="demonstrate every capability")
context.add_component("message_bus", BUS)

runs = []
for iteration in range(2):
    context.scratch["iteration"] = iteration
    result = showcase_agent.run(context)
    context.push_message(result.out_messages[-1])
    runs.append(
        {
            "iteration": iteration,
            "temperature": context.scratch["context"]["parameters"]["temperature"],
            "tool_result": context.scratch["showcase_metrics"]["tool_result"],
            "llm_preview": result.out_messages[-1].content[:80],
        }
    )

runs


[{'iteration': 0,
  'temperature': 0.7999999999999999,
  'tool_result': 20.0,
  'llm_preview': "[stub llm] ters: {'temperature': 0.7, 'tone': 'friendly'}Latest tool output: 20."},
 {'iteration': 1,
  'temperature': 0.8999999999999999,
  'tool_result': 20.0,
  'llm_preview': "[stub llm] ture': 0.7999999999999999, 'tone': 'friendly'}Latest tool output: 20."}]

In [5]:
# Cell 4: inspect aggregated state across capabilities

{
    "inbox": [message.content[:80] for message in context.inbox],
    "memory_events": context.memory["default"].search(""),
    "registered_skills": list(context.registry.list().keys()),
    "available_tools": list(context.components["tools"].keys()),
    "rl_journal": context.scratch.get("showcase_rl", []),
    "final_parameters": dict(context.scratch["context"]["parameters"]),
    "message_bus": [
        {"agent": record.agent, "channel": record.channel, "content": record.content[:80]}
        for record in BUS.conversation()
    ],
}


{'inbox': ["[stub llm] ters: {'temperature': 0.7, 'tone': 'friendly'}Latest tool output: 20.",
  "[stub llm] ture': 0.7999999999999999, 'tone': 'friendly'}Latest tool output: 20."],
 'memory_events': [{'key': 'last_greeting',
   'value': 'Greetings, demonstrate every capability!'},
  {'key': 'last_greeting',
   'value': 'Greetings, demonstrate every capability!'}],
 'registered_skills': ['greeting'],
 'available_tools': ['math'],
 'rl_journal': [{'iteration': 0,
   'tool_result': 20.0,
   'target': 25.0,
   'action': 'increase',
   'temperature': 0.8},
  {'iteration': 1,
   'tool_result': 20.0,
   'target': 25.0,
   'action': 'increase',
   'temperature': 0.9}],
 'final_parameters': {'temperature': 0.8999999999999999, 'tone': 'friendly'},
 'message_bus': [{'agent': 'showcase_agent',
   'channel': 'summary',
   'content': "[stub llm] ters: {'temperature': 0.7, 'tone': 'friendly'}Latest tool output: 20."},
  {'agent': 'rl_showcase',
   'channel': 'rl',
   'content': '[iter 0] increase te